# 🛒 IndiaKart E-Commerce Analytics — End-to-End Pipeline
### **Internship Project Submission | Data Analytics Domain**
**Company / Platform:** Internmo | IndiaKart Marketplace  
**Batch:** August 2026 – November 2026  
**Analyst:** Data Analyst Intern  
**Dataset Scope:** 24 Months Operations (June 2023 – June 2025) | 50,000 Orders · ~222,000 Records across 8 Interconnected Tables  
**Deliverables:** Phase 1 (Data Cleaning & Quality Audit), Phase 2 (10 Core EDA Visualizations), Phase 3 (Business KPI Scorecard), Phase 4 (Executive Summary & Strategic Plan)



---
## ⚙️ 1. Environment Setup & Library Imports
We initialize standard data analytics libraries (`pandas`, `numpy`, `matplotlib`, `seaborn`) and configure high-contrast corporate plotting styles.



In [ ]:
# Environment & Library Setup
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# Inline plotting for Google Colab / Jupyter
%matplotlib inline

# Style configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#94A3B8'
plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['grid.color'] = '#E2E8F0'
plt.rcParams['grid.linestyle'] = '--'
plt.rcParams['grid.alpha'] = 0.7

print("✓ Libraries imported and visualization settings configured successfully.")


---
## 📥 2. Ingesting All 8 Relational Datasets
We load the 8 interconnected CSV tables into memory. The script automatically detects whether it is running in Google Colab (`/content/`), locally, or in the workspace environment.



In [ ]:
# Define search directories for data files
candidate_dirs = [
    '/workspace/indiakart_project',
    '/content/indiakart_project',
    '/content',
    './indiakart_project',
    '.'
]

DATA_DIR = None
for d in candidate_dirs:
    if os.path.exists(os.path.join(d, 'orders DA.csv')):
        DATA_DIR = d
        break

if DATA_DIR is None:
    raise FileNotFoundError("Could not locate 'orders DA.csv'. Please upload project CSV files.")

print(f"Loading datasets from: {DATA_DIR}")

customers = pd.read_csv(os.path.join(DATA_DIR, 'customers DA.csv'))
orders = pd.read_csv(os.path.join(DATA_DIR, 'orders DA.csv'))
order_items = pd.read_csv(os.path.join(DATA_DIR, 'order_items DA.csv'))
payments = pd.read_csv(os.path.join(DATA_DIR, 'payments DA.csv'))
returns = pd.read_csv(os.path.join(DATA_DIR, 'returns DA.csv'))
products = pd.read_csv(os.path.join(DATA_DIR, 'products DA.csv'))
inventory = pd.read_csv(os.path.join(DATA_DIR, 'inventory DA.csv'))
suppliers = pd.read_csv(os.path.join(DATA_DIR, 'suppliers DA.csv'))

datasets = {
    'customers': customers,
    'orders': orders,
    'order_items': order_items,
    'payments': payments,
    'returns': returns,
    'products': products,
    'inventory': inventory,
    'suppliers': suppliers
}

print("\nDataset Summary:")
for name, df in datasets.items():
    print(f"  • {name:<12}: {df.shape[0]:>7,} rows × {df.shape[1]:>2} columns")


---
## 🧹 Phase 1: Data Cleaning & Quality Assurance Audit
A thorough data audit examining:
1. **Primary Key Uniqueness** across all 8 tables.
2. **Referential Integrity** between orders, customers, items, payments, and returns.
3. **Date Chronology Validation** (Ensuring `delivered_date >= order_date`).
4. **Calculated Amount Consistency** (`final_amount = total_amount + gst_amount + shipping_charge - discount_amount`).
5. **Artifact Flagging** (Documenting 10,071 orders with no line item rows without silent row dropping).



In [ ]:
# Phase 1: Comprehensive Data Quality & Integrity Checks

# 1. Primary Key Uniqueness
pk_dict = {
    'customers': 'customer_id',
    'orders': 'order_id',
    'order_items': 'item_id',
    'payments': 'payment_id',
    'returns': 'return_id',
    'products': 'product_id',
    'inventory': 'inventory_id',
    'suppliers': 'supplier_id'
}

print("1. Primary Key Uniqueness Check:")
for tbl, pk in pk_dict.items():
    dups = datasets[tbl][pk].duplicated().sum()
    print(f"  • {tbl:<12} ({pk}): {dups} duplicates found {'✓' if dups == 0 else '❌'}")

# 2. Date Parsing & Chronology Validation
orders['order_date'] = pd.to_datetime(orders['order_date'], dayfirst=True)
orders['delivered_date'] = pd.to_datetime(orders['delivered_date'], dayfirst=True)
orders['order_year_month'] = orders['order_date'].dt.to_period('M').astype(str)

deliv_mask = orders['delivered_date'].notna()
lead_days = (orders.loc[deliv_mask, 'delivered_date'] - orders.loc[deliv_mask, 'order_date']).dt.days
invalid_dates = (lead_days < 0).sum()

print(f"\n2. Date Chronology Validation:")
print(f"  • Delivered orders checked: {deliv_mask.sum():,}")
print(f"  • Invalid delivery dates (delivered < ordered): {invalid_dates} {'✓' if invalid_dates == 0 else '❌'}")
print(f"  • Average delivery lead time: {lead_days.mean():.2f} days (Min: {lead_days.min()} days, Max: {lead_days.max()} days)")

# 3. Referential Integrity Check
cust_set = set(customers['customer_id'])
ord_set = set(orders['order_id'])
prod_set = set(products['product_id'])

orphan_orders = (~orders['customer_id'].isin(cust_set)).sum()
orphan_items = (~order_items['order_id'].isin(ord_set)).sum()
orphan_payments = (~payments['order_id'].isin(ord_set)).sum()
orphan_returns = (~returns['order_id'].isin(ord_set)).sum()

print(f"\n3. Referential Integrity (Foreign Keys):")
print(f"  • Orphan orders (missing customer): {orphan_orders} {'✓' if orphan_orders == 0 else '❌'}")
print(f"  • Orphan items (missing order): {orphan_items} {'✓' if orphan_items == 0 else '❌'}")
print(f"  • Orphan payments (missing order): {orphan_payments} {'✓' if orphan_payments == 0 else '❌'}")
print(f"  • Orphan returns (missing order): {orphan_returns} {'✓' if orphan_returns == 0 else '❌'}")

# 4. Synthetic Data Artifact Audit
orders_with_items = set(order_items['order_id'])
orders_no_items = (~orders['order_id'].isin(orders_with_items)).sum()
print(f"\n4. Data Artifact Audit:")
print(f"  • Flagged orders without item lines: {orders_no_items:,} ({orders_no_items/len(orders)*100:.2f}%)")
print(f"    (Documented as a synthetic artifact; all orders retained for order-level analyses).")


---
## 📊 Phase 2: Exploratory Data Analysis (All 10 Visualizations)
Below we generate the complete set of 10 business visualizations mandated by the IndiaKart Project PRD.



### Chart 1: Monthly Order Volume (June 2023 – June 2025)
Highlights the seasonal surge during the annual Diwali shopping festival (October & November).



In [ ]:
monthly_orders = orders.groupby('order_year_month').size().reset_index(name='order_count')

plt.figure(figsize=(12, 5), dpi=150)
bar_colors = ['#D97706' if m.split('-')[1] in ['10', '11'] else '#1E3A8A' for m in monthly_orders['order_year_month']]
bars = plt.bar(monthly_orders['order_year_month'], monthly_orders['order_count'], color=bar_colors, width=0.65)

plt.title('Chart 1: Monthly Order Volume (June 2023 – June 2025)\n[Amber bars highlight Diwali Festive Window (Oct & Nov)]', fontsize=12, fontweight='bold', pad=12, color='#0F172A')
plt.xlabel('Year-Month', fontweight='bold', labelpad=8)
plt.ylabel('Total Orders Placed', fontweight='bold', labelpad=8)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))
plt.tight_layout()
plt.show()


### Chart 2: Monthly Gross Revenue (GMV) Trend
Demonstrates monthly financial throughput (in ₹ Crores) across the 24-month operational period.



In [ ]:
monthly_gmv = orders.groupby('order_year_month')['final_amount'].sum().reset_index()
monthly_gmv['gmv_crores'] = monthly_gmv['final_amount'] / 1e7

plt.figure(figsize=(12, 5), dpi=150)
plt.plot(monthly_gmv['order_year_month'], monthly_gmv['gmv_crores'], marker='o', color='#0F766E', linewidth=2.5, markersize=6)
plt.fill_between(monthly_gmv['order_year_month'], monthly_gmv['gmv_crores'], color='#0F766E', alpha=0.15)

plt.title('Chart 2: Monthly GMV Trend (₹ Crores)\nSteady baseline with marked peak in Diwali Season (Nov 2023: ₹15.89 Cr)', fontsize=12, fontweight='bold', pad=12, color='#0F172A')
plt.xlabel('Year-Month', fontweight='bold', labelpad=8)
plt.ylabel('Gross Merchandise Value (₹ Crores)', fontweight='bold', labelpad=8)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.tight_layout()
plt.show()


### Chart 3: Category-wise Revenue Share
Reveals substantial revenue concentration in **Electronics** (~57.4% of total item revenue).



In [ ]:
cat_rev = order_items.groupby('category')['total_price'].sum().sort_values(ascending=True)
cat_rev_cr = cat_rev / 1e7

plt.figure(figsize=(10, 6), dpi=150)
colors = ['#334155'] * (len(cat_rev) - 1) + ['#E11D48']  # Highlight Electronics in Crimson
bars = plt.barh(cat_rev.index, cat_rev_cr, color=colors, height=0.65)

plt.title('Chart 3: Category Revenue Share (₹ Crores)\nElectronics dominates at ~57.4% (₹178.6 Cr), driving category concentration risk', fontsize=12, fontweight='bold', pad=12, color='#0F172A')
plt.xlabel('Revenue (₹ Crores)', fontweight='bold', labelpad=8)

for bar in bars:
    w = bar.get_width()
    pct = (w / cat_rev_cr.sum()) * 100
    plt.text(w + 2, bar.get_y() + bar.get_height()/2, f"₹{w:.1f} Cr ({pct:.1f}%)", va='center', fontsize=9, fontweight='bold', color='#1E293B')

plt.xlim(0, max(cat_rev_cr) * 1.25)
plt.tight_layout()
plt.show()


### Chart 4: Order Status Distribution
Donut chart illustrating overall fulfillment rates and the structural cancellation rate (11.99%).



In [ ]:
status_counts = orders['status'].value_counts()

plt.figure(figsize=(7, 7), dpi=150)
donut_colors = ['#0F766E', '#E11D48', '#0284C7', '#D97706', '#334155']
wedges, texts, autotexts = plt.pie(
    status_counts, 
    labels=status_counts.index, 
    autopct='%1.1f%%', 
    startangle=140, 
    colors=donut_colors[:len(status_counts)],
    wedgeprops=dict(width=0.4, edgecolor='white', linewidth=2),
    pctdistance=0.75
)
plt.setp(autotexts, size=9, weight="bold", color="white")
plt.setp(texts, size=10, weight="bold")
plt.title('Chart 4: Order Status Distribution\n65.0% Delivered | 12.0% Structural Cancellation Rate', fontsize=12, fontweight='bold', pad=12, color='#0F172A')
plt.tight_layout()
plt.show()


### Chart 5: Top 10 States by Number of Orders
Examines geographic concentration: Uttar Pradesh and Maharashtra represent the core user hubs.



In [ ]:
top_states = orders[orders['status'] != 'Cancelled']['state'].value_counts().head(10).sort_values(ascending=True)

plt.figure(figsize=(10, 6), dpi=150)
bars = plt.barh(top_states.index, top_states.values, color='#1E3A8A', height=0.65)
plt.title('Chart 5: Top 10 States by Active Orders\nUttar Pradesh and Maharashtra lead; Top-5 states represent ~57.8% of volume', fontsize=12, fontweight='bold', pad=12, color='#0F172A')
plt.xlabel('Number of Active Orders', fontweight='bold', labelpad=8)

for bar in bars:
    plt.text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2, f"{bar.get_width():,}", va='center', fontsize=9, fontweight='bold', color='#1E293B')

plt.xlim(0, max(top_states.values) * 1.18)
plt.tight_layout()
plt.show()


### Chart 6: Customer Segment Distribution
Breaks down the 10,000 registered users across **Regular** (50.0%), **Budget** (30.0%), and **Premium** (20.0%) segments.



In [ ]:
seg_counts = customers['segment'].value_counts()

plt.figure(figsize=(7, 5), dpi=150)
bars = plt.bar(seg_counts.index, seg_counts.values, color=['#0F766E', '#0284C7', '#D97706'], width=0.55)
plt.title('Chart 6: Customer Segment Distribution\nRegular Segment dominates user base (50.0%), followed by Budget & Premium', fontsize=12, fontweight='bold', pad=12, color='#0F172A')
plt.ylabel('Number of Registered Customers', fontweight='bold', labelpad=8)

for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, h + 80, f"{h:,}\n({h/len(customers)*100:.1f}%)", ha='center', fontsize=9, fontweight='bold', color='#1E293B')

plt.ylim(0, max(seg_counts.values) * 1.18)
plt.tight_layout()
plt.show()


### Chart 7: Payment Method Usage & Failure Rate
Dual-axis chart evaluating transaction popularity alongside gateway failure rates (Benchmark target: ≤ 2.0%).



In [ ]:
pay_methods = payments.groupby('payment_method').agg(
    total_tx=('payment_id', 'count'),
    failed_tx=('status', lambda s: (s == 'Failed').sum())
).sort_values('total_tx', ascending=False)
pay_methods['failure_rate'] = (pay_methods['failed_tx'] / pay_methods['total_tx']) * 100

fig, ax1 = plt.subplots(figsize=(10, 5), dpi=150)
ax2 = ax1.twinx()

bars = ax1.bar(pay_methods.index, pay_methods['total_tx'], color='#334155', width=0.5, label='Transactions')
line = ax2.plot(pay_methods.index, pay_methods['failure_rate'], color='#E11D48', marker='o', linewidth=2.5, label='Failure Rate %')

ax1.set_title('Chart 7: Payment Method Adoption vs Failure Rate (%)\nCredit Card & UPI lead usage; Overall failure rate of 3.50% exceeds the 2.0% target', fontsize=12, fontweight='bold', pad=12, color='#0F172A')
ax1.set_ylabel('Transaction Count', fontweight='bold', labelpad=8, color='#334155')
ax2.set_ylabel('Failure Rate (%)', fontweight='bold', labelpad=8, color='#E11D48')
ax2.set_ylim(0, 5)
ax2.axhline(2.0, color='#10B981', linestyle='--', linewidth=1.5, label='Target Max (2.0%)')
ax1.yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))

plt.tight_layout()
plt.show()


### Chart 8: Customer Age Distribution Histogram
Analyzes the age demographic of shoppers (Median age: ~36 years).



In [ ]:
plt.figure(figsize=(10, 5), dpi=150)
n, bins, patches = plt.hist(customers['age'], bins=20, color='#0284C7', edgecolor='white', linewidth=1.2)
plt.axvline(customers['age'].median(), color='#E11D48', linestyle='--', linewidth=2, label=f"Median Age: {customers['age'].median():.0f} yrs")

plt.title('Chart 8: Customer Age Distribution\nCore shopper demographic centered around 25–45 age bracket', fontsize=12, fontweight='bold', pad=12, color='#0F172A')
plt.xlabel('Customer Age (Years)', fontweight='bold', labelpad=8)
plt.ylabel('Customer Count', fontweight='bold', labelpad=8)
plt.legend(frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.show()


### Chart 9: Breakdown of Return Reasons
Assesses the drivers of customer returns: Size & Fit issues and Transit Damage account for nearly half of all complaints.



In [ ]:
ret_reasons = returns['reason'].value_counts().sort_values(ascending=True)

plt.figure(figsize=(10, 5), dpi=150)
bars = plt.barh(ret_reasons.index, ret_reasons.values, color='#D97706', height=0.6)
plt.title('Chart 9: Breakdown of Return Reasons\nSize & Fit, Mind Changed, and Transit Damage drive >60% of all customer returns', fontsize=12, fontweight='bold', pad=12, color='#0F172A')
plt.xlabel('Number of Return Requests', fontweight='bold', labelpad=8)

for bar in bars:
    plt.text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2, f"{bar.get_width():,} ({bar.get_width()/len(returns)*100:.1f}%)", va='center', fontsize=9, fontweight='bold')

plt.xlim(0, max(ret_reasons.values) * 1.22)
plt.tight_layout()
plt.show()


### Chart 10: Average Order Value (AOV) by Customer Segment
Validates segment monetization: Premium shoppers generate significantly higher basket values.



In [ ]:
merged_orders = orders[orders['status'] != 'Cancelled'].merge(customers[['customer_id', 'segment']], on='customer_id')
aov_segment = merged_orders.groupby('segment')['final_amount'].mean().reindex(['Budget', 'Regular', 'Premium'])

plt.figure(figsize=(8, 5), dpi=150)
bars = plt.bar(aov_segment.index, aov_segment.values, color=['#64748B', '#0284C7', '#0F766E'], width=0.5)
plt.title('Chart 10: Average Order Value (AOV) by Customer Segment\nPremium customers yield significantly higher basket values (₹64k+)', fontsize=12, fontweight='bold', pad=12, color='#0F172A')
plt.ylabel('Average Order Value (₹)', fontweight='bold', labelpad=8)

for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, h + 1000, f"₹{h:,.0f}", ha='center', fontsize=10, fontweight='bold', color='#0F172A')

plt.ylim(0, max(aov_segment.values) * 1.15)
plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('₹{x:,.0f}'))
plt.tight_layout()
plt.show()


---
## 🎯 Phase 3: Core Business KPI Calculations (Verified Scorecard)
We compute the 10 foundational operational and commercial KPIs against industry benchmarks.



In [ ]:
# Phase 3: Business KPI Calculations
active_orders = orders[orders['status'] != 'Cancelled']
delivered_orders = orders[orders['status'] == 'Delivered']

# 1. GMV
active_gmv = float(active_orders['final_amount'].sum())
delivered_gmv = float(delivered_orders['final_amount'].sum())

# 2. AOV
active_aov = float(active_orders['final_amount'].mean())

# 3. Cancellation Rate
cancellation_rate = float((orders['status'] == 'Cancelled').mean() * 100)

# 4. Payment Failure Rate
payment_failure_rate = float((payments['status'] == 'Failed').mean() * 100)

# 5. Return Rate on Delivered Orders
return_rate_delivered = float(len(returns) / len(delivered_orders) * 100)

# 6. Inventory In-Stock Rate
inventory_fill_rate = float((inventory['status'] == 'In Stock').mean() * 100)

# 7. Category Concentration (Electronics)
elec_share = float(order_items[order_items['category'] == 'Electronics']['total_price'].sum() / order_items['total_price'].sum() * 100)

# 8. Top 5 States Share
top5_share = float(active_orders['state'].value_counts().head(5).sum() / len(active_orders) * 100)

# 9. App Channel Share
app_share = float((active_orders['channel'] == 'App').mean() * 100)

scorecard = pd.DataFrame([
    {"KPI / Metric": "Total Orders Analysed", "Benchmark / Target": "N/A", "Actual Achieved": f"{len(orders):,}", "Status": "Complete"},
    {"KPI / Metric": "Active Orders (Excl. Cancelled)", "Benchmark / Target": "N/A", "Actual Achieved": f"{len(active_orders):,}", "Status": "88.01% Active"},
    {"KPI / Metric": "Delivered Orders", "Benchmark / Target": "N/A", "Actual Achieved": f"{len(delivered_orders):,}", "Status": "65.00% Fulfilled"},
    {"KPI / Metric": "Active GMV (Gross Merchandise Value)", "Benchmark / Target": "₹250.0 Cr", "Actual Achieved": f"₹{active_gmv/1e7:.2f} Cr", "Status": "Above Target (+11.2%)"},
    {"KPI / Metric": "Delivered GMV", "Benchmark / Target": "N/A", "Actual Achieved": f"₹{delivered_gmv/1e7:.2f} Cr", "Status": "73.9% Realized"},
    {"KPI / Metric": "Average Order Value (Active AOV)", "Benchmark / Target": "₹50,000", "Actual Achieved": f"₹{active_aov:,.2f}", "Status": "Healthy Basket"},
    {"KPI / Metric": "Order Cancellation Rate", "Benchmark / Target": "< 8.00%", "Actual Achieved": f"{cancellation_rate:.2f}%", "Status": "Watch (11.99%)"},
    {"KPI / Metric": "Payment Failure Rate", "Benchmark / Target": "≤ 2.00%", "Actual Achieved": f"{payment_failure_rate:.2f}%", "Status": "Exceeds Target (3.50%)"},
    {"KPI / Metric": "Return Rate on Delivered Orders", "Benchmark / Target": "≤ 20.00%", "Actual Achieved": f"{return_rate_delivered:.2f}%", "Status": "High (30.77%)"},
    {"KPI / Metric": "Inventory In-Stock / Fill Rate", "Benchmark / Target": "≥ 95.00%", "Actual Achieved": f"{inventory_fill_rate:.2f}%", "Status": "Below Target (92.20%)"},
    {"KPI / Metric": "Electronics Category Revenue Share", "Benchmark / Target": "< 40.00%", "Actual Achieved": f"{elec_share:.2f}%", "Status": "High Concentration"},
    {"KPI / Metric": "Top 5 States Geographic Share", "Benchmark / Target": "< 45.00%", "Actual Achieved": f"{top5_share:.2f}%", "Status": "Concentrated in UP & MH"},
    {"KPI / Metric": "Mobile App Channel Share", "Benchmark / Target": "> 50.00%", "Actual Achieved": f"{app_share:.2f}%", "Status": "Primary Channel (55.02%)"}
])

print("="*85)
print("                       INDIAKART VERIFIED KPI SCORECARD")
print("="*85)
print(scorecard.to_string(index=False))
print("="*85)


---
## 📑 Phase 4: Executive Management Summary & Strategic Recommendations

### Executive Summary (Addressed to CEO)
IndiaKart processed **50,000 orders** over the 24-month observation window (June 2023 – June 2025), generating **₹277.91 Crores in Active GMV** (**₹205.39 Crores Delivered**). Average Order Value (AOV) is robust at **₹63,156**, with the Mobile App serving as the primary commercial channel (**55.02%** share of active orders). The business displays steady baseline performance punctuated by strong annual surges during the Diwali festive window.

### Key Operational Risks
1. **Payment Failure Rate (3.50% vs 2.0% Benchmark):** Checkout friction accounts for ~1,748 dropped transactions.
2. **Category Concentration Risk (57.41% Electronics):** Heavy reliance on Electronics leaves the marketplace vulnerable to consumer electronics cycle dips.
3. **Elevated Return Rate (30.77% of Delivered Orders):** High reverse logistics overhead driven by sizing issues (28%) and transit damage (20%).
4. **Sub-optimal Inventory Fill Rate (92.20% vs 95.0% Target):** 76 Low Stock SKUs and 2 Out of Stock SKUs creating lost revenue opportunities.

### Actionable Strategic Plan
* **Action 1 (Payment Reliability):** Integrate smart gateway cascading (e.g. Razorpay + Cashfree) with automatic retry logic for failed UPI transactions and pre-validated VPA checks.
* **Action 2 (Category Diversification):** Aggressively onboard high-margin vendors in Sports & Fitness, Home & Kitchen, and Fashion to bring Electronics share below 45%.
* **Action 3 (Reverse Logistics Reduction):** Deploy interactive 3D sizing charts for fashion categories and mandate reinforced transit packaging for long-distance Tier 2/3 delivery routes.
* **Action 4 (Automated Replenishment):** Establish dynamic buffer stock thresholds linked to festive lead-times and supplier credit terms.
* **Action 5 (App Loyalty Program):** Convert Regular shoppers (50% of user base) to Premium via app-exclusive reward tiers and targeted basket-building discounts.

